In [ ]:
import pandas
import numpy

In [ ]:
df = pandas.read_csv("income_data.csv", skipinitialspace=True)

# Laplace mechanism

In [ ]:
df['age'].plot.hist()

In [ ]:
# how many individuals are over 50 years of age
(df['age'] > 50).sum()

In [ ]:
# let's construct a Laplace mechanism
sensitivity = 1
epsilon = 0.1
(df['age'] > 50).sum() + numpy.random.laplace(0, sensitivity/epsilon)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
draws = (df['age'] > 50).sum() + numpy.random.laplace(0, sensitivity/epsilon, size=5000)
plt.hist(draws, bins=25)
plt.xlabel('Count of individuals aged > 50')
plt.ylabel('# results')
None

In [ ]:
df['target'] # 1 is high, 0 is low

In [ ]:
# what's the fraction of > 50 year olds in high income bracket
(df['target'][df['age'] > 50]).mean()

In [ ]:
df['target'].mean()

In [ ]:
df['age'][df['target'] == 0].plot.hist(alpha=0.5)
df['age'][df['target'] == 1].plot.hist(alpha=0.5)

In [ ]:
# one step in decision tree
for split in sorted(df['age'].unique()):
    mask = df['age'] < split
    gini = 1 - df['target'][mask].mean()**2 - df['target'][~mask].mean()**2
    print(split, gini)

# Exponential Mechanism

In [ ]:
df['education'].unique() # R

In [ ]:
df['education'].value_counts()

In [ ]:
# scoring function for the query 'most common education'
def u(x, value):
    v_cnts = x.value_counts()
    return v_cnts[value] / v_cnts.sum()

In [ ]:
u(df['education'], 'Some-college')

In [ ]:
df['education'].drop(2)

In [ ]:
df['education'].shape[0]

In [ ]:
answers = []
for _ in range(1000):
    drop = numpy.random.choice(df['education'].shape[0])
    xp = df['education'].drop(drop)
    answers.append(numpy.abs(u(df['education'], 'Some-college') - u(xp, 'Some-college')))

In [ ]:
numpy.unique(answers)

In [ ]:
u(df['education'], numpy.array(['Some-college', 'HS-grad']))

In [ ]:
def exponential(x, R, u, sensitivity, epsilon, n=1):
    scores = u(x, R) # score each element in R
    probs = numpy.exp(epsilon*scores / 2 / sensitivity)
    probs /= probs.sum()
    return numpy.random.choice(R, n, p=probs)

In [ ]:
exponential(df['education'], df['education'].unique(), u, 1, 1)

In [ ]:
import pandas

In [ ]:
pandas.Series(exponential(df['education'], df['education'].unique(), u, 1, 4, 10000)
             ).value_counts().plot.bar()

In [ ]:
df['education'].value_counts().plot.bar()

# Differential Privacy Library

https://diffprivlib.readthedocs.io/en/latest/index.html

In [ ]:
df.columns

In [ ]:
from diffprivlib.models import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score

In [ ]:
def mk_model(epsilon):
    return Pipeline([
        ('process', ColumnTransformer([
            ('one-hot', OneHotEncoder(drop='first', sparse=False), ['education', 'sex']), # careful!
            ('leave-age', FunctionTransformer(), ['age'])
        ])),
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(data_norm=50, epsilon=epsilon))
    ])

In [ ]:
mk_model(0.5).fit(df, df['target'])

In [ ]:
eps = numpy.linspace(0.1, 2, 10)
scores = [cross_val_score(mk_model(e), df, df['target']) for e in eps]

In [ ]:
plt.plot(eps, scores)
plt.xlabel('epsilon')
plt.ylabel('cross validation scores')
None